# S46_03 — Chain-of-Thought Reasoning

Chain-of-thought (CoT) prompting instructs the model to reason step-by-step before giving a final answer. It significantly improves accuracy on arithmetic, logic, and multi-step tasks — with no additional training.

## Why CoT works

LLMs predict tokens sequentially. When the model generates intermediate reasoning steps, those tokens become context for the final answer — effectively giving the model a scratchpad. This unlocks capabilities that are suppressed when the model must answer in one token.

In [ ]:
import anthropic

client = anthropic.Anthropic()

def ask(system, user, model='claude-haiku-4-5-20251001', max_tokens=512):
    msg = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system,
        messages=[{'role': 'user', 'content': user}],
    )
    return msg.content[0].text

problem = (
    'A train leaves city A at 9:00am travelling at 80 mph. '
    'A second train leaves city B (320 miles away) at 10:00am travelling toward city A at 60 mph. '
    'At what time do they meet?'
)

# Without CoT
direct = ask('You are a helpful assistant.', problem + ' Give only the answer.')
print('Direct answer:', direct)

# With CoT
cot = ask(
    'You are a helpful assistant.',
    problem + ' Think step by step, then give the final answer.'
)
print('\nChain-of-thought:')
print(cot)

## Zero-shot CoT

Simply adding *"Think step by step"* (or *"Let's think step by step"*) to a prompt triggers CoT reasoning without any examples. Works across model families.

In [ ]:
# Zero-shot CoT variants — all effective
triggers = [
    'Think step by step.',
    "Let's think through this carefully.",
    'Work through this problem step by step before answering.',
    'First, reason about this. Then give your answer.',
]

q = 'If I have 3 apples and give half to my friend, then buy 4 more, how many do I have?'

for trigger in triggers[:2]:   # run 2 to save tokens
    result = ask('You are a helpful assistant.', q + ' ' + trigger)
    print(f'Trigger: "{trigger}"')
    print(result)
    print()

## Few-shot CoT — exemplars with reasoning

In [ ]:
few_shot_cot = """
Solve each word problem. Show your reasoning, then state the answer.

Q: A store has 24 apples. They sell 1/3 in the morning and 1/4 of the remainder in the afternoon. How many are left?
A: Start with 24 apples.
   Morning: sell 24 × 1/3 = 8. Remaining: 24 - 8 = 16.
   Afternoon: sell 16 × 1/4 = 4. Remaining: 16 - 4 = 12.
   Answer: 12 apples.

Q: A tank fills in 6 hours with pipe A, and 4 hours with pipe B. How long to fill together?
A: Pipe A fills 1/6 per hour, pipe B fills 1/4 per hour.
   Combined rate: 1/6 + 1/4 = 2/12 + 3/12 = 5/12 per hour.
   Time = 12/5 = 2.4 hours = 2 hours 24 minutes.
   Answer: 2 hours 24 minutes.

Q: A bag has 5 red and 3 blue marbles. Two marbles are drawn without replacement. What is the probability both are red?
A:"""

result = ask('You are a helpful assistant.', few_shot_cot)
print(result)

## Self-consistency — majority vote over multiple CoT paths

Sample the same problem multiple times (temperature > 0), extract the final answer from each, and take the majority. More reliable than a single sample.

In [ ]:
import re
from collections import Counter

def ask_n(user, n=5, temperature=0.7):
    answers = []
    for _ in range(n):
        msg = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=256,
            temperature=temperature,
            messages=[{'role': 'user', 'content': user}],
        )
        answers.append(msg.content[0].text)
    return answers

def extract_final_number(text):
    numbers = re.findall(r'\b(\d+(?:\.\d+)?)\b', text)
    return numbers[-1] if numbers else None

q = (
    'Roger has 5 tennis balls. He buys 2 more cans of tennis balls. '
    'Each can has 3 balls. How many tennis balls does he have now? '
    'Think step by step, then write "Answer: X" at the end.'
)

responses = ask_n(q, n=5)
answers = [extract_final_number(r) for r in responses]
print('Individual answers:', answers)
vote = Counter(answers).most_common(1)[0]
print(f'Majority vote: {vote[0]} ({vote[1]}/5 agree)')  # should be 11

## Structured CoT with XML tags

For Claude models, using XML-style `<thinking>` tags forces the model to separate reasoning from output — useful when you need a clean final answer separate from scratchpad.

In [ ]:
import re

result = ask(
    system='You are a precise reasoning assistant. Always show your work in <thinking> tags, then give the final answer in <answer> tags.',
    user='What is 17% of 250, rounded to the nearest integer?'
)
print(result)

# Extract just the answer
answer_match = re.search(r'<answer>(.*?)</answer>', result, re.DOTALL)
if answer_match:
    print('\nExtracted answer:', answer_match.group(1).strip())

## When to use CoT

| Use CoT | Skip CoT |
|---------|----------|
| Multi-step arithmetic / logic | Simple classification |
| Symbolic reasoning | Single-fact lookup |
| Planning / decomposition | Sentiment analysis |
| Code debugging | Named entity extraction |
| Causal inference | Direct retrieval tasks |

**Cost:** CoT generates more output tokens, which costs more. Use it where accuracy gains justify the extra cost.

Next: [S46_04_llm_apis.ipynb](./S46_04_llm_apis.ipynb)